# PhoBERT — tiền xử lý và fine-tuning trên ViFactCheck

## Mục tiêu
Notebook xây dựng mô hình phân loại quan hệ giữa **Statement** (phát biểu cần kiểm chứng) và **Evidence** (bằng chứng). Mô hình học dự đoán một trong ba nhãn số `0`, `1`, `2` của bộ dữ liệu ViFactCheck.

## Quy trình thực hiện
**Đọc dữ liệu đã làm sạch chung → xử lý URL → tách từ tiếng Việt → mã hóa cặp văn bản → huấn luyện PhoBERT → đánh giá và lưu kết quả.**

Dữ liệu đầu vào nằm trong `data/processed/common_cleaned/`, gồm ba tập: train để học tham số, dev để chọn checkpoint và test để đánh giá sau khi chọn mô hình. Notebook giữ chữ hoa, dấu câu và từ phủ định. Cột `Context` được giữ trong CSV nhưng không được đưa vào mô hình; thí nghiệm này sử dụng cặp `Statement + Evidence` có sẵn trong dữ liệu.

Chạy các cell theo thứ tự từ trên xuống. VnCoreNLP cần Java; việc tải tài nguyên và PhoBERT cần Internet khi chưa có trong môi trường. Biến `RUN_TRAINING` điều khiển việc thực hiện phần huấn luyện.

### Chuẩn bị thư viện

Cài các thư viện đọc dữ liệu, xử lý tensor, sử dụng PhoBERT, tách từ tiếng Việt và tính metric.

In [1]:
# Chạy một lần nếu môi trường chưa có thư viện.
%pip install pandas numpy torch transformers sentencepiece scikit-learn py_vncorenlp tqdm

Note: you may need to restart the kernel to use updated packages.


### Khai báo thư viện, đường dẫn và cấu hình

Hàm `find_project_root` xác định thư mục chứa dữ liệu từ vị trí chạy notebook. Các đường dẫn đầu vào, dữ liệu xử lý, tài nguyên tách từ và mô hình đều được xây dựng từ thư mục gốc này. `MAX_LENGTH` giới hạn độ dài cặp token; `REMOVE_URLS` điều khiển việc loại URL.

In [2]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import os
import random
import re
import shutil

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             classification_report, confusion_matrix)

def find_project_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "data/processed/common_cleaned").is_dir():
            return candidate
    # Hỗ trợ mở notebook từ NLP hoặc folder giải nén bên ngoài.
    matches = [f.parent.parent.parent for f in start.glob(
        "CS221-NLP-Project-Vi-Fact-Checking-main/**/data/processed/common_cleaned")]
    matches = sorted(set(matches))
    if len(matches) == 1:
        return matches[0]
    raise FileNotFoundError("Hãy chạy notebook từ project chứa data/processed/common_cleaned.")

PROJECT_ROOT = find_project_root(Path.cwd())
INPUT_DIR = PROJECT_ROOT / "data/processed/common_cleaned"
BASE_DIR = PROJECT_ROOT / "data/processed/phobert"
MODEL_DIR = PROJECT_ROOT / "notebooks/models/Phobert"
VNCORENLP_DIR = PROJECT_ROOT / "resources/vncorenlp"
BASE_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "vinai/phobert-base-v2"
MAX_LENGTH = 256
REMOVE_URLS = True  # Loại URL khỏi Statement và Evidence trước khi tách từ.
RUN_TRAINING = True  # Đặt True để thực hiện phần huấn luyện.
SPLITS = ("train", "dev", "test")
print("Project:", PROJECT_ROOT)
print("Input:", INPUT_DIR)
print("Dữ liệu PhoBERT:", BASE_DIR)

/opt/homebrew/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/CS221-NLP-Project-Vi-Fact-Checking-main/CS221-NLP-Project-Vi-Fact-Checking-main
Input: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/CS221-NLP-Project-Vi-Fact-Checking-main/CS221-NLP-Project-Vi-Fact-Checking-main/data/processed/common_cleaned
Dữ liệu PhoBERT: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/CS221-NLP-Project-Vi-Fact-Checking-main/CS221-NLP-Project-Vi-Fact-Checking-main/data/processed/phobert


## 1. Đọc và kiểm tra dữ liệu đầu vào

Mỗi tập được đọc từ file `vifactcheck_<split>_common_cleaned.csv`. Chương trình kiểm tra sự tồn tại của file, các cột bắt buộc, văn bản thiếu hoặc rỗng và miền giá trị nhãn. Nếu phát hiện dữ liệu không hợp lệ, chương trình dừng để người thực hiện kiểm tra thay vì tự loại bỏ mẫu.

Thứ tự dòng và cách chia train/dev/test được giữ nguyên. Mã SHA-256 của từng file được lưu để xác định chính xác nguồn dữ liệu của thí nghiệm. Bảng thống kê cho biết số mẫu và phân bố nhãn của mỗi tập. Nhãn được giữ ở dạng số theo dữ liệu đầu vào.

In [3]:
frames = {}
source_hashes = {}
for split in SPLITS:
    path = INPUT_DIR / f"vifactcheck_{split}_common_cleaned.csv"
    if not path.is_file():
        raise FileNotFoundError(f"Thiếu {path}. Chạy notebooks/preprocessing/common_cleaning.ipynb trước.")
    source_hashes[split] = hashlib.sha256(path.read_bytes()).hexdigest()
    df = pd.read_csv(path)
    required = {"Statement", "Evidence", "labels"}
    if not required.issubset(df.columns):
        raise ValueError(f"{split}: thiếu cột {required - set(df.columns)}")
    for col in ("Statement", "Evidence"):
        if df[col].isna().any() or df[col].astype(str).str.strip().eq("").any():
            raise ValueError(f"{split}: {col} chứa dữ liệu thiếu/rỗng.")
    if df["labels"].isna().any() or not df["labels"].isin([0, 1, 2]).all():
        raise ValueError(f"{split}: nhãn phải thuộc 0, 1, 2.")
    df["labels"] = df["labels"].astype(int)
    frames[split] = df
pd.DataFrame([{ "split": s, "rows": len(df),
                "label_counts": df["labels"].value_counts().sort_index().to_dict()}
              for s, df in frames.items()])

,split,rows,label_counts
0,train,5062,"{0: 1751, 1: 1658, 2: 1653}"
1,dev,723,"{0: 256, 1: 244, 2: 223}"
2,test,1447,"{0: 508, 1: 468, 2: 471}"


### 1.1. Xử lý URL cho đầu vào PhoBERT

Hàm `prepare_phobert_text` thay URL bắt đầu bằng `http://`, `https://` hoặc `www.` bằng khoảng trắng khi `REMOVE_URLS=True`, sau đó chuẩn hóa khoảng trắng. Mục đích là loại chuỗi địa chỉ web khỏi văn bản đưa vào mô hình. Regex dùng `\S+` nên có thể lấy cả dấu câu liền sau URL; đây là đặc điểm của quy tắc xử lý cần lưu ý khi diễn giải dữ liệu. Chương trình kiểm tra văn bản rỗng và lưu bản sao đã xử lý; file common cleaned không bị thay đổi.

In [4]:
URL_PATTERN = re.compile(r"https?://\S+|www\.\S+", flags=re.IGNORECASE)

def prepare_phobert_text(text):
    text = str(text)
    if REMOVE_URLS:
        text = URL_PATTERN.sub(" ", text)
    return re.sub(r"\s+", " ", text).strip()

prepared = {}
for split, df in frames.items():
    result = df.copy()
    for col in ("Statement", "Evidence"):
        result[col] = result[col].map(prepare_phobert_text)
        if result[col].eq("").any():
            raise ValueError(f"{split}: {col} rỗng sau xử lý URL; cần kiểm tra thủ công.")
    result.to_csv(BASE_DIR / f"vifactcheck_{split}_phobert_cleaned.csv", index=False, encoding="utf-8-sig")
    prepared[split] = result
prepared["train"][["Statement", "Evidence", "labels"]].head(3)

,Statement,Evidence,labels
0,"Phó Thủ tướng Trần Hồng Hà thay mặt Chính phủ,...","Thay mặt Chính phủ, Thủ tướng Chính phủ, Phó T...",0
1,Hành vi của Tô Văn Hải là cho phép người khác ...,Tô Văn Hải đã có hành vi cho phép người khác đ...,0
2,SAWACO thông báo tạm ngưng cung cấp nước để th...,SAWACO thông báo tạm ngưng cung cấp nước để th...,1


## 2. Tách từ tiếng Việt bằng VnCoreNLP

Trong tiếng Việt, một từ có thể gồm nhiều âm tiết cách nhau bởi khoảng trắng. Bước tách từ xác định các âm tiết thuộc cùng một từ và biểu diễn chúng bằng dấu gạch dưới, ví dụ `sinh_viên`, trước khi đưa vào tokenizer PhoBERT.

Chương trình kiểm tra Java và các tài nguyên VnCoreNLP trong `resources/vncorenlp/`, tải tài nguyên nếu thiếu và khởi tạo bộ tách từ với tác vụ `wseg`. Mỗi văn bản được tách từ theo câu, sau đó ghép lại thành một chuỗi. Kết quả rỗng được xem là lỗi cần kiểm tra.

Chỉ xử lý `Statement` và `Evidence`. Các cột metadata, nhãn và `Context` được giữ nguyên. Kết quả từng tập được lưu thành file `*_phobert_segmented.csv`.

In [5]:
import py_vncorenlp

if shutil.which("java") is None:
    raise RuntimeError("Chưa tìm thấy Java. Cài Java và khởi động lại kernel trước khi chạy VnCoreNLP.")
VNCORENLP_DIR.mkdir(parents=True, exist_ok=True)
required_resources = ["VnCoreNLP-1.2.jar", "models/wordsegmenter/wordsegmenter.rdr", "models/wordsegmenter/vi-vocab"]
if not all((VNCORENLP_DIR / name).is_file() for name in required_resources):
    py_vncorenlp.download_model(save_dir=str(VNCORENLP_DIR))
if not all((VNCORENLP_DIR / name).is_file() for name in required_resources):
    raise FileNotFoundError(f"Tài nguyên VnCoreNLP chưa đầy đủ: {VNCORENLP_DIR}")
segmenter = py_vncorenlp.VnCoreNLP(annotators=["wseg"], save_dir=str(VNCORENLP_DIR))

def word_segment(text):
    result = " ".join(segmenter.word_segment(text))
    if not result.strip():
        raise ValueError("VnCoreNLP trả về chuỗi rỗng cho văn bản hợp lệ.")
    return result

tqdm.pandas(desc="Word segmentation")
segmented = {}
for split, df in prepared.items():
    result = df.copy()
    for col in ("Statement", "Evidence"):
        print(split, col)
        result[col] = result[col].progress_apply(word_segment)
    result.to_csv(BASE_DIR / f"vifactcheck_{split}_phobert_segmented.csv", index=False, encoding="utf-8-sig")
    segmented[split] = result
segmented["train"][["Statement", "Evidence", "labels"]].head(3)

2026-09-15 23:25:21 INFO  WordSegmenter:24 - Loading Word Segmentation model
train Statement


Word segmentation: 100%|██████████| 5062/5062 [00:02<00:00, 1869.36it/s]


train Evidence


Word segmentation: 100%|██████████| 5062/5062 [00:02<00:00, 2051.75it/s]


dev Statement


...

test Statement


Word segmentation: 100%|██████████| 1447/1447 [00:00<00:00, 2493.23it/s]


test Evidence


Word segmentation: 100%|██████████| 1447/1447 [00:00<00:00, 2119.86it/s]


,Statement,Evidence,labels
0,Phó Thủ_tướng Trần_Hồng_Hà thay_mặt Chính_phủ ...,"Thay_mặt Chính_phủ , Thủ_tướng Chính_phủ , Phó...",0
1,Hành_vi của Tô_Văn_Hải là cho_phép người khác ...,Tô_Văn_Hải đã có hành_vi cho_phép người khác đ...,0
2,SAWACO thông_báo tạm ngưng cung_cấp nước để th...,SAWACO thông_báo tạm ngưng cung_cấp nước để th...,1


## 3. Mã hóa cặp văn bản và lưu tensor

Tokenizer của `vinai/phobert-base-v2` chuyển văn bản đã tách từ thành mã token. Thứ tự đầu vào là **Statement trước, Evidence sau**; tokenizer bổ sung các token đặc biệt để biểu diễn cặp văn bản.

Mỗi mẫu có tối đa 256 token, tính cả token đặc biệt. Với cặp dài hơn giới hạn, `longest_first` cắt token từ thành phần đang dài hơn; với cặp ngắn hơn, padding bổ sung token đệm. Bảng thống kê đếm số mẫu bị cắt để theo dõi khả năng mất thông tin.

Ba tensor được lưu cho mỗi tập:
- `input_ids`: mã token, kích thước `(số mẫu, 256)`.
- `attention_mask`: đánh dấu token nội dung/token đặc biệt và vị trí đệm.
- `labels`: nhãn đích, kích thước `(số mẫu,)`.

Mapping nhãn được tạo từ train và áp dụng thống nhất cho dev/test. Tokenizer cùng cấu hình tiền xử lý, mã kiểm tra dữ liệu nguồn và thống kê token được lưu để tái lập thí nghiệm và chuẩn bị đầu vào khi suy luận.

In [6]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)
train_unique_labels = sorted(int(v) for v in segmented["train"]["labels"].unique())
if train_unique_labels != [0, 1, 2]:
    raise ValueError("Train cần có đủ ba nhãn 0, 1, 2.")
label2id = {label: idx for idx, label in enumerate(train_unique_labels)}
id2label = {idx: label for label, idx in label2id.items()}
token_stats = []
for split, df in segmented.items():
    statements = df["Statement"].tolist()
    evidences = df["Evidence"].tolist()
    unpadded = tokenizer(statements, evidences, padding=False, truncation=False)
    lengths = [len(ids) for ids in unpadded["input_ids"]]
    encoded = tokenizer(statements, evidences, padding="max_length", truncation="longest_first",
                        max_length=MAX_LENGTH, return_attention_mask=True, return_tensors="pt")
    labels = torch.tensor(df["labels"].map(label2id).tolist(), dtype=torch.long)
    data = {"input_ids": encoded["input_ids"], "attention_mask": encoded["attention_mask"], "labels": labels}
    assert data["input_ids"].shape == data["attention_mask"].shape == (len(df), MAX_LENGTH)
    assert labels.shape == (len(df),)
    torch.save(data, BASE_DIR / f"vifactcheck_{split}_phobert_tokenized.pt")
    token_stats.append({"split": split, "rows": len(df), "truncated_rows": sum(n > MAX_LENGTH for n in lengths),
                        "max_tokens_before_truncation": max(lengths)})
# Chỉ chứa kiểu Python cơ bản, đọc được với weights_only=True.
torch.save({"label2id": label2id, "id2label": id2label}, BASE_DIR / "phobert_label_mapping.pt")
TOKENIZER_DIR = BASE_DIR / "tokenizer"
tokenizer.save_pretrained(TOKENIZER_DIR)
manifest = {"model_name": MODEL_NAME, "max_length": MAX_LENGTH, "remove_urls": REMOVE_URLS,
            "pair_order": ["Statement", "Evidence"], "source_sha256": source_hashes,
            "label2id": label2id, "token_stats": token_stats,
            "created_at": datetime.now(timezone.utc).isoformat()}
(BASE_DIR / "preprocessing_config.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
pd.DataFrame(token_stats)

[transformers] Be aware, overflowing tokens are not returned for the setting you have chosen, i.e. sequence pairs with the 'longest_first' truncation strategy. So the returned list will always be empty even if some tokens have been removed.


,split,rows,truncated_rows,max_tokens_before_truncation
0,train,5062,0,237
1,dev,723,0,223
2,test,1447,1,276


## 4. Fine-tune PhoBERT

Phần này cập nhật trọng số PhoBERT cho bài toán phân loại ba nhãn từ cặp Statement–Evidence. Đặt `RUN_TRAINING=True` để thực hiện; nếu biến bằng `False`, notebook kết thúc sau phần tiền xử lý bằng thông báo `SystemExit`.

Mô hình được huấn luyện 5 epoch. Sau mỗi epoch, chương trình đánh giá trên dev và lưu checkpoint có **Macro-F1 cao nhất**. Tập test chỉ được sử dụng để đánh giá sau khi checkpoint đã được chọn.

Mỗi lần thực nghiệm có một thư mục `phobert_model/runs/<thời gian>/` riêng. Checkpoint được lưu cùng tokenizer và cấu hình tiền xử lý để sử dụng thống nhất khi suy luận.

In [7]:
if not RUN_TRAINING:
    raise SystemExit("Đã hoàn thành tiền xử lý. Đặt RUN_TRAINING=True nếu muốn chạy phần fine-tuning.")

### 4.1. Siêu tham số huấn luyện

Thiết lập batch size 16, 5 epoch, learning rate 2e-5, weight decay 0.01, tỷ lệ warmup 10% và giới hạn chuẩn gradient bằng 1.0. Seed 42 được dùng để giảm biến động giữa các lần chạy.

In [8]:
MODEL_NAME = "vinai/phobert-base-v2"
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1
MAX_GRAD_NORM = 1.0
RANDOM_SEED = 42

### 4.2. Chọn thiết bị tính toán

Ưu tiên MPS trên máy Apple, tiếp theo là CUDA và cuối cùng là CPU. Tensor và mô hình được chuyển tới cùng thiết bị khi huấn luyện và đánh giá.

In [9]:
print("\n" + "=" * 70)
print("DEVICE")
print("=" * 70)
if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print("Using device:", DEVICE)


DEVICE
Using device: mps


### 4.3. Thiết lập seed

Cố định seed cho Python, NumPy và PyTorch. Thiết lập này hỗ trợ tái lập kết quả, nhưng không bảo đảm kết quả giống tuyệt đối giữa mọi thiết bị hoặc phiên bản thư viện.

In [10]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(RANDOM_SEED)
print("Random seed:", RANDOM_SEED)

Random seed: 42


### 4.4. Xác định file huấn luyện

Sử dụng các tensor và mapping nhãn được tạo ở phần tiền xử lý.

In [11]:
train_file = os.path.join(
    BASE_DIR,
    "vifactcheck_train_phobert_tokenized.pt"
)
dev_file = os.path.join(
    BASE_DIR,
    "vifactcheck_dev_phobert_tokenized.pt"
)
test_file = os.path.join(
    BASE_DIR,
    "vifactcheck_test_phobert_tokenized.pt"
)
label_mapping_file = os.path.join(
    BASE_DIR,
    "phobert_label_mapping.pt"
)

### 4.5. Tạo thư mục thực nghiệm

Mã lần chạy được tạo từ thời gian hiện tại. Mỗi lần thực nghiệm có thư mục riêng để lưu checkpoint, lịch sử huấn luyện và dự đoán.

In [12]:
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
OUTPUT_DIR = MODEL_DIR / "phobert_model" / "runs" / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
BEST_MODEL_DIR = OUTPUT_DIR / "best_model"
print("Training output:", OUTPUT_DIR)

Training output: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/CS221-NLP-Project-Vi-Fact-Checking-main/CS221-NLP-Project-Vi-Fact-Checking-main/notebooks/models/Phobert/phobert_model/runs/20260915_232534_858342


### 4.6. Kiểm tra và nạp tensor

Kiểm tra đủ file trước khi đọc. Tensor được nạp vào CPU; mỗi batch sẽ được chuyển sang thiết bị tính toán khi cần.

In [13]:
print("\n" + "=" * 70)
print("CHECK INPUT FILES")
print("=" * 70)
input_files = {
    "Train": train_file,
    "Dev": dev_file,
    "Test": test_file,
    "Label mapping": label_mapping_file
}
for name, path in input_files.items():
    if not os.path.isfile(path):
        raise FileNotFoundError(
            f"\n{name} file không tồn tại:\n{path}"
        )
    print(f"{name}: OK")


CHECK INPUT FILES
Train: OK
Dev: OK
Test: OK
Label mapping: OK


In [14]:
print("\n" + "=" * 70)
print("LOADING TOKENIZED DATA")
print("=" * 70)
train_data = torch.load(
    train_file,
    map_location="cpu",
    weights_only=True
)
dev_data = torch.load(
    dev_file,
    map_location="cpu",
    weights_only=True
)
test_data = torch.load(
    test_file,
    map_location="cpu",
    weights_only=True
)
label_mapping = torch.load(
    label_mapping_file,
    map_location="cpu",
    weights_only=True
)
print("Train data loaded successfully.")
print("Dev data loaded successfully.")
print("Test data loaded successfully.")
print("Label mapping loaded successfully.")


LOADING TOKENIZED DATA
Train data loaded successfully.
Dev data loaded successfully.
Test data loaded successfully.
Label mapping loaded successfully.


### 4.7. Đọc mapping và kiểm tra kích thước

Lấy mapping nhãn và số lớp. Kiểm tra các khóa bắt buộc và số mẫu giữa input_ids, attention_mask và labels để tránh lệch đầu vào với nhãn.

In [15]:
label2id = label_mapping["label2id"]
id2label = label_mapping["id2label"]
print("\nLabel mapping:")
print("label2id:")
for label, idx in label2id.items():
    print(
        f"  {label} -> {idx}"
    )
print("\nid2label:")
for idx, label in id2label.items():
    print(
        f"  {idx} -> {label}"
    )
NUM_LABELS = len(label2id)
print(
    "\nNumber of labels:",
    NUM_LABELS
)


Label mapping:
label2id:
  0 -> 0
  1 -> 1
  2 -> 2

id2label:
  0 -> 0
  1 -> 1
  2 -> 2

Number of labels: 3


In [16]:
print("\n" + "=" * 70)
print("CHECK TOKENIZED DATA")
print("=" * 70)
def check_tokenized_data(data, name):
    required_keys = [
        "input_ids",
        "attention_mask",
        "labels"
    ]
    for key in required_keys:
        if key not in data:
            raise ValueError(
                f"{name} thiếu key: {key}"
            )
    print(f"\n{name}")
    print(
        "  input_ids      :",
        data["input_ids"].shape
    )
    print(
        "  attention_mask :",
        data["attention_mask"].shape
    )
    print(
        "  labels         :",
        data["labels"].shape
    )
    n_input = data["input_ids"].shape[0]
    n_mask = data["attention_mask"].shape[0]
    n_labels = data["labels"].shape[0]
    if not (
        n_input == n_mask == n_labels
    ):
        raise ValueError(
            f"{name}: số lượng input_ids, "
            f"attention_mask và labels không giống nhau."
        )
    print(
        "  Number samples :",
        n_input
    )
check_tokenized_data(
    train_data,
    "TRAIN"
)
check_tokenized_data(
    dev_data,
    "DEV"
)
check_tokenized_data(
    test_data,
    "TEST"
)


CHECK TOKENIZED DATA

TRAIN
  input_ids      : torch.Size([5062, 256])
  attention_mask : torch.Size([5062, 256])
  labels         : torch.Size([5062])
  Number samples : 5062

DEV
  input_ids      : torch.Size([723, 256])
  attention_mask : torch.Size([723, 256])
  labels         : torch.Size([723])
  Number samples : 723

TEST
  input_ids      : torch.Size([1447, 256])
  attention_mask : torch.Size([1447, 256])
  labels         : torch.Size([1447])
  Number samples : 1447


### 4.8. Xây dựng Dataset và DataLoader

Lớp `PhoBERTDataset` trả về các tensor của một mẫu. `DataLoader` chia dữ liệu thành batch; chỉ train được xáo trộn, còn dev/test giữ thứ tự để đối chiếu dự đoán.

In [17]:
class PhoBERTDataset(Dataset):
    def __init__(self, data):
        self.input_ids = data["input_ids"]
        self.attention_mask = data["attention_mask"]
        self.labels = data["labels"]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        return {
            "input_ids":
                self.input_ids[idx],
            "attention_mask":
                self.attention_mask[idx],
            "labels":
                self.labels[idx]
        }

In [18]:
train_dataset = PhoBERTDataset(
    train_data
)
dev_dataset = PhoBERTDataset(
    dev_data
)
test_dataset = PhoBERTDataset(
    test_data
)
print("\n" + "=" * 70)
print("DATASET SIZE")
print("=" * 70)
print(
    "Train:",
    len(train_dataset)
)
print(
    "Dev  :",
    len(dev_dataset)
)
print(
    "Test :",
    len(test_dataset)
)


DATASET SIZE
Train: 5062
Dev  : 723
Test : 1447


In [19]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)
dev_loader = DataLoader(
    dev_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)
print("\n" + "=" * 70)
print("DATALOADER")
print("=" * 70)
print(
    "Batch size:",
    BATCH_SIZE
)
print(
    "Train batches:",
    len(train_loader)
)
print(
    "Dev batches:",
    len(dev_loader)
)
print(
    "Test batches:",
    len(test_loader)
)


DATALOADER
Batch size: 16
Train batches: 317
Dev batches: 46
Test batches: 91


### 4.9. Khởi tạo mô hình phân loại

Nạp trọng số pretrained của PhoBERT và cấu hình đầu phân loại có ba lớp. Bước fine-tuning học các tham số cho nhiệm vụ kiểm chứng phát biểu.

In [20]:
print("\n" + "=" * 70)
print("LOADING PHOBERT MODEL")
print("=" * 70)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label={
        int(k): str(v)
        for k, v in id2label.items()
    },
    label2id={
        str(k): int(v)
        for k, v in label2id.items()
    }
)


LOADING PHOBERT MODEL


Loading weights: 100%|██████████| 197/197 [00:00<00:00, 33941.75it/s]
[transformers] RobertaForSequenceClassification LOAD REPORT from: vinai/phobert-base-v2
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.bias    | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [21]:
model.to(DEVICE)
print(
    "Model loaded:",
    MODEL_NAME
)
print(
    "Number of labels:",
    NUM_LABELS
)
print(
    "Device:",
    DEVICE
)

Model loaded: vinai/phobert-base-v2
Number of labels: 3
Device: mps


### 4.10. Optimizer và lịch learning rate

AdamW cập nhật tham số theo gradient và áp dụng weight decay. Learning rate tăng trong giai đoạn warmup, sau đó giảm tuyến tính theo tổng số bước huấn luyện.

In [22]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

In [23]:
total_training_steps = (
    len(train_loader) * EPOCHS
)
warmup_steps = int(
    total_training_steps * WARMUP_RATIO
)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps
)
print("\n" + "=" * 70)
print("OPTIMIZER / SCHEDULER")
print("=" * 70)
print(
    "Learning rate:",
    LEARNING_RATE
)
print(
    "Weight decay:",
    WEIGHT_DECAY
)
print(
    "Total training steps:",
    total_training_steps
)
print(
    "Warmup steps:",
    warmup_steps
)


OPTIMIZER / SCHEDULER
Learning rate: 2e-05
Weight decay: 0.01
Total training steps: 1585
Warmup steps: 158


### 4.11. Các chỉ số đánh giá

Accuracy là tỷ lệ dự đoán đúng. Precision, Recall và F1 được tính theo macro: tính riêng cho từng lớp rồi lấy trung bình, giúp mỗi lớp có trọng số ngang nhau trong chỉ số tổng hợp.

In [24]:
def calculate_metrics(
    y_true,
    y_pred
):
    accuracy = accuracy_score(
        y_true,
        y_pred
    )
    precision, recall, f1, _ = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )
    )
    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

### 4.12. Huấn luyện một epoch

Bật chế độ train, xóa gradient, tính loss từ nhãn đích, lan truyền ngược, giới hạn chuẩn gradient rồi cập nhật optimizer và scheduler. Dự đoán là lớp có logit lớn nhất. Hàm trả về metric toàn epoch và loss trung bình theo batch.

In [25]:
def train_one_epoch(
    model,
    loader,
    optimizer,
    scheduler,
    device
):
    model.train()
    total_loss = 0.0
    all_predictions = []
    all_labels = []
    for batch_idx, batch in enumerate(loader):
        input_ids = batch[
            "input_ids"
        ].to(device)
        attention_mask = batch[
            "attention_mask"
        ].to(device)
        labels = batch[
            "labels"
        ].to(device)
        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        loss = outputs.loss
        logits = outputs.logits
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM
        )
        optimizer.step()
        scheduler.step()
        total_loss += loss.item()
        predictions = torch.argmax(
            logits,
            dim=1
        )
        all_predictions.extend(
            predictions.detach()
            .cpu()
            .numpy()
        )
        all_labels.extend(
            labels.detach()
            .cpu()
            .numpy()
        )
        if (
            batch_idx + 1
        ) % 100 == 0:
            print(
                f"  Batch "
                f"{batch_idx + 1}/"
                f"{len(loader)}"
                f" - Loss: "
                f"{loss.item():.4f}"
            )
    average_loss = (
        total_loss /
        len(loader)
    )
    metrics = calculate_metrics(
        all_labels,
        all_predictions
    )
    metrics["loss"] = average_loss
    return metrics

### 4.13. Đánh giá và chọn checkpoint

Hàm `evaluate` dùng chế độ eval và tắt tính gradient. Vòng lặp thực hiện train rồi đánh giá dev ở mỗi epoch, ghi lịch sử và lưu trọng số khi dev Macro-F1 đạt giá trị cao nhất tính đến thời điểm đó. Test không tham gia chọn checkpoint. Loss báo cáo là trung bình loss của các batch.

In [26]:
def evaluate(
    model,
    loader,
    device
):
    model.eval()
    total_loss = 0.0
    all_predictions = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            input_ids = batch[
                "input_ids"
            ].to(device)
            attention_mask = batch[
                "attention_mask"
            ].to(device)
            labels = batch[
                "labels"
            ].to(device)
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )
            loss = outputs.loss
            logits = outputs.logits
            total_loss += loss.item()
            predictions = torch.argmax(
                logits,
                dim=1
            )
            all_predictions.extend(
                predictions.cpu().numpy()
            )
            all_labels.extend(
                labels.cpu().numpy()
            )
    average_loss = (
        total_loss /
        len(loader)
    )
    metrics = calculate_metrics(
        all_labels,
        all_predictions
    )
    metrics["loss"] = average_loss
    return (
        metrics,
        all_labels,
        all_predictions
    )
print("\n" + "=" * 70)
print("START TRAINING")
print("=" * 70)
print(
    "Epochs:",
    EPOCHS
)
print(
    "Batch size:",
    BATCH_SIZE
)
print(
    "Learning rate:",
    LEARNING_RATE
)
best_dev_f1 = -1.0
best_epoch = 0
training_history = []
for epoch in range(
    1,
    EPOCHS + 1
):
    print("\n")
    print("=" * 70)
    print(
        f"EPOCH {epoch}/{EPOCHS}"
    )
    print("=" * 70)
    print("\n[TRAIN]")
    train_metrics = train_one_epoch(
        model,
        train_loader,
        optimizer,
        scheduler,
        DEVICE
    )
    print(
        f"Train Loss     : "
        f"{train_metrics['loss']:.4f}"
    )
    print(
        f"Train Accuracy : "
        f"{train_metrics['accuracy']:.4f}"
    )
    print(
        f"Train Precision: "
        f"{train_metrics['precision']:.4f}"
    )
    print(
        f"Train Recall   : "
        f"{train_metrics['recall']:.4f}"
    )
    print(
        f"Train F1       : "
        f"{train_metrics['f1']:.4f}"
    )
    print("\n[DEV]")
    dev_metrics, _, _ = evaluate(
        model,
        dev_loader,
        DEVICE
    )
    print(
        f"Dev Loss       : "
        f"{dev_metrics['loss']:.4f}"
    )
    print(
        f"Dev Accuracy   : "
        f"{dev_metrics['accuracy']:.4f}"
    )
    print(
        f"Dev Precision  : "
        f"{dev_metrics['precision']:.4f}"
    )
    print(
        f"Dev Recall     : "
        f"{dev_metrics['recall']:.4f}"
    )
    print(
        f"Dev F1         : "
        f"{dev_metrics['f1']:.4f}"
    )
    training_history.append({
        "epoch": epoch,
        "train_loss":
            train_metrics["loss"],
        "train_accuracy":
            train_metrics["accuracy"],
        "train_precision":
            train_metrics["precision"],
        "train_recall":
            train_metrics["recall"],
        "train_f1":
            train_metrics["f1"],
        "dev_loss":
            dev_metrics["loss"],
        "dev_accuracy":
            dev_metrics["accuracy"],
        "dev_precision":
            dev_metrics["precision"],
        "dev_recall":
            dev_metrics["recall"],
        "dev_f1":
            dev_metrics["f1"]
    })
    if dev_metrics["f1"] > best_dev_f1:
        best_dev_f1 = (
            dev_metrics["f1"]
        )
        best_epoch = epoch
        print("\n*** NEW BEST MODEL ***")
        print(
            f"Best Dev F1: "
            f"{best_dev_f1:.4f}"
        )
        os.makedirs(
            BEST_MODEL_DIR,
            exist_ok=True
        )
        model.save_pretrained(
            BEST_MODEL_DIR
        )
        torch.save(
            {
                "epoch": epoch,
                "best_dev_f1":
                    best_dev_f1,
                "label2id":
                    label2id,
                "id2label":
                    id2label,
                "model_name":
                    MODEL_NAME,
                "max_length":
                    256,
                "batch_size":
                    BATCH_SIZE,
                "learning_rate":
                    LEARNING_RATE,
                "weight_decay":
                    WEIGHT_DECAY
            },
            os.path.join(
                BEST_MODEL_DIR,
                "training_config.pt"
            )
        )
# Lưu tokenizer và chính sách preprocessing cùng checkpoint đã chọn.
tokenizer.save_pretrained(BEST_MODEL_DIR)
(BEST_MODEL_DIR / "preprocessing_config.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


START TRAINING
Epochs: 5
Batch size: 16
Learning rate: 2e-05


EPOCH 1/5

[TRAIN]
  Batch 100/317 - Loss: 1.1101
  Batch 200/317 - Loss: 0.8664
  Batch 300/317 - Loss: 0.6548
Train Loss     : 0.8908
Train Accuracy : 0.5731
Train Precision: 0.5742
Train Recall   : 0.5707
Train F1       : 0.5689

[DEV]
Dev Loss       : 0.5665
Dev Accuracy   : 0.7911
Dev Precision  : 0.7939
Dev Recall     : 0.7940
Dev F1         : 0.7911

*** NEW BEST MODEL ***
Best Dev F1: 0.7911


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.73it/s]




EPOCH 2/5

[TRAIN]
  Batch 100/317 - Loss: 0.4983
  Batch 200/317 - Loss: 0.4810
  Batch 300/317 - Loss: 0.3271
Train Loss     : 0.4541
Train Accuracy : 0.8420
Train Precision: 0.8416
Train Recall   : 0.8416
Train F1       : 0.8415

[DEV]
Dev Loss       : 0.5325
Dev Accuracy   : 0.7994
Dev Precision  : 0.8097
Dev Recall     : 0.7965
Dev F1         : 0.7995

*** NEW BEST MODEL ***
Best Dev F1: 0.7995


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.46it/s]




EPOCH 3/5

[TRAIN]
  Batch 100/317 - Loss: 0.2349
  Batch 200/317 - Loss: 0.3479
  Batch 300/317 - Loss: 0.4785
Train Loss     : 0.3054
Train Accuracy : 0.9044
Train Precision: 0.9044
Train Recall   : 0.9043
Train F1       : 0.9043

[DEV]
Dev Loss       : 0.6183
Dev Accuracy   : 0.8091
Dev Precision  : 0.8216
Dev Recall     : 0.8098
Dev F1         : 0.8110

*** NEW BEST MODEL ***
Best Dev F1: 0.8110


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.30it/s]




EPOCH 4/5

[TRAIN]
  Batch 100/317 - Loss: 0.0359
  Batch 200/317 - Loss: 0.0294
  Batch 300/317 - Loss: 0.0239
Train Loss     : 0.1951
Train Accuracy : 0.9439
Train Precision: 0.9439
Train Recall   : 0.9438
Train F1       : 0.9439

[DEV]
Dev Loss       : 0.6427
Dev Accuracy   : 0.8368
Dev Precision  : 0.8389
Dev Recall     : 0.8382
Dev F1         : 0.8364

*** NEW BEST MODEL ***
Best Dev F1: 0.8364


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.43it/s]




EPOCH 5/5

[TRAIN]
  Batch 100/317 - Loss: 0.0268
  Batch 200/317 - Loss: 0.1003
  Batch 300/317 - Loss: 0.0205
Train Loss     : 0.1388
Train Accuracy : 0.9644
Train Precision: 0.9646
Train Recall   : 0.9644
Train F1       : 0.9645

[DEV]
Dev Loss       : 0.6879
Dev Accuracy   : 0.8340
Dev Precision  : 0.8340
Dev Recall     : 0.8358
Dev F1         : 0.8341


912

### 4.14. Nạp checkpoint được chọn

In epoch và dev Macro-F1 tốt nhất, sau đó nạp lại checkpoint đã lưu để thực hiện đánh giá cuối cùng.

In [27]:
print("\n" + "=" * 70)
print("TRAINING FINISHED")
print("=" * 70)
print(
    "Best epoch:",
    best_epoch
)
print(
    "Best Dev F1:",
    f"{best_dev_f1:.4f}"
)


TRAINING FINISHED
Best epoch: 4
Best Dev F1: 0.8364


In [28]:
print("\n" + "=" * 70)
print("LOADING BEST MODEL")
print("=" * 70)
best_model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        BEST_MODEL_DIR
    )
)
best_model.to(DEVICE)


LOADING BEST MODEL


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6036.18it/s]


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(64001, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (position_embeddings): Embedding(258, 768, padding_idx=1)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
  

### 4.15. Đánh giá trên dev và test

Tính loss, Accuracy, Macro-Precision, Macro-Recall và Macro-F1 của checkpoint đã chọn trên từng tập. Dev dùng để đối chiếu kết quả chọn mô hình; test cung cấp kết quả đánh giá sau huấn luyện.

In [29]:
print("\n" + "=" * 70)
print("FINAL DEV EVALUATION")
print("=" * 70)
dev_metrics, dev_true, dev_pred = evaluate(
    best_model,
    dev_loader,
    DEVICE
)
print(
    f"Dev Loss      : "
    f"{dev_metrics['loss']:.4f}"
)
print(
    f"Dev Accuracy  : "
    f"{dev_metrics['accuracy']:.4f}"
)
print(
    f"Dev Precision : "
    f"{dev_metrics['precision']:.4f}"
)
print(
    f"Dev Recall    : "
    f"{dev_metrics['recall']:.4f}"
)
print(
    f"Dev F1        : "
    f"{dev_metrics['f1']:.4f}"
)


FINAL DEV EVALUATION
Dev Loss      : 0.6427
Dev Accuracy  : 0.8368
Dev Precision : 0.8389
Dev Recall    : 0.8382
Dev F1        : 0.8364


In [30]:
print("\n" + "=" * 70)
print("FINAL TEST EVALUATION")
print("=" * 70)
test_metrics, test_true, test_pred = evaluate(
    best_model,
    test_loader,
    DEVICE
)
print(
    f"Test Loss      : "
    f"{test_metrics['loss']:.4f}"
)
print(
    f"Test Accuracy  : "
    f"{test_metrics['accuracy']:.4f}"
)
print(
    f"Test Precision : "
    f"{test_metrics['precision']:.4f}"
)
print(
    f"Test Recall    : "
    f"{test_metrics['recall']:.4f}"
)
print(
    f"Test F1        : "
    f"{test_metrics['f1']:.4f}"
)


FINAL TEST EVALUATION
Test Loss      : 0.5893
Test Accuracy  : 0.8473
Test Precision : 0.8479
Test Recall    : 0.8465
Test F1        : 0.8463


### 4.16. Phân tích kết quả theo lớp

Classification report trình bày Precision, Recall, F1 và số mẫu của từng nhãn. Ma trận nhầm lẫn có hàng là nhãn thực tế và cột là nhãn dự đoán, giúp nhận diện các cặp nhãn dễ bị nhầm.

In [31]:
print("\n" + "=" * 70)
print("TEST CLASSIFICATION REPORT")
print("=" * 70)
target_names = [
    str(id2label[i])
    for i in range(NUM_LABELS)
]
print(
    classification_report(
        test_true,
        test_pred,
        labels=list(
            range(NUM_LABELS)
        ),
        target_names=target_names,
        digits=4,
        zero_division=0
    )
)


TEST CLASSIFICATION REPORT
              precision    recall  f1-score   support

           0     0.8296    0.8720    0.8503       508
           1     0.8498    0.7735    0.8098       468
           2     0.8645    0.8938    0.8789       471

    accuracy                         0.8473      1447
   macro avg     0.8479    0.8465    0.8463      1447
weighted avg     0.8475    0.8473    0.8465      1447



In [32]:
print("\n" + "=" * 70)
print("TEST CONFUSION MATRIX")
print("=" * 70)
cm = confusion_matrix(
    test_true,
    test_pred,
    labels=list(
        range(NUM_LABELS)
    )
)
print(cm)


TEST CONFUSION MATRIX
[[443  41  24]
 [ 64 362  42]
 [ 27  23 421]]


### 4.17. Lưu kết quả và tổng kết

Xuất lịch sử huấn luyện theo epoch và dự đoán test thành CSV. Các cột tên nhãn hiện biểu diễn nhãn số gốc qua mapping. Cell tổng kết hiển thị checkpoint được chọn, metric test và đường dẫn các file kết quả.

In [33]:
history_file = os.path.join(
    OUTPUT_DIR,
    "training_history.csv"
)
import pandas as pd
history_df = pd.DataFrame(
    training_history
)
history_df.to_csv(
    history_file,
    index=False
)
print("\nTraining history saved:")
print(history_file)


Training history saved:
/Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/CS221-NLP-Project-Vi-Fact-Checking-main/CS221-NLP-Project-Vi-Fact-Checking-main/notebooks/models/Phobert/phobert_model/runs/20260915_232534_858342/training_history.csv


In [34]:
predictions_file = os.path.join(
    OUTPUT_DIR,
    "test_predictions.csv"
)
prediction_df = pd.DataFrame({
    "true_label":
        test_true,
    "predicted_label":
        test_pred
})
prediction_df[
    "true_label_name"
] = prediction_df[
    "true_label"
].map(
    id2label
)
prediction_df[
    "predicted_label_name"
] = prediction_df[
    "predicted_label"
].map(
    id2label
)
prediction_df.to_csv(
    predictions_file,
    index=False
)
print(
    "Test predictions saved:"
)
print(
    predictions_file
)

Test predictions saved:
/Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/CS221-NLP-Project-Vi-Fact-Checking-main/CS221-NLP-Project-Vi-Fact-Checking-main/notebooks/models/Phobert/phobert_model/runs/20260915_232534_858342/test_predictions.csv


In [35]:
print("\n" + "=" * 70)
print("PHOBERT TRAINING COMPLETED!")
print("=" * 70)
print("\nModel:")
print(MODEL_NAME)
print("\nBest epoch:")
print(best_epoch)
print("\nBest Dev F1:")
print(
    f"{best_dev_f1:.4f}"
)
print("\nFinal Test Results:")
print(
    f"  Accuracy  : "
    f"{test_metrics['accuracy']:.4f}"
)
print(
    f"  Precision : "
    f"{test_metrics['precision']:.4f}"
)
print(
    f"  Recall    : "
    f"{test_metrics['recall']:.4f}"
)
print(
    f"  F1        : "
    f"{test_metrics['f1']:.4f}"
)
print("\nSaved model:")
print(BEST_MODEL_DIR)
print("\nSaved files:")
print(
    "  Model:",
    BEST_MODEL_DIR
)
print(
    "  History:",
    history_file
)
print(
    "  Predictions:",
    predictions_file
)
print("\n" + "=" * 70)
print("DONE")
print("=" * 70)


PHOBERT TRAINING COMPLETED!

Model:
vinai/phobert-base-v2

Best epoch:
4

Best Dev F1:
0.8364

Final Test Results:
  Accuracy  : 0.8473
  Precision : 0.8479
  Recall    : 0.8465
  F1        : 0.8463

Saved model:
/Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/CS221-NLP-Project-Vi-Fact-Checking-main/CS221-NLP-Project-Vi-Fact-Checking-main/notebooks/models/Phobert/phobert_model/runs/20260915_232534_858342/best_model

Saved files:
  Model: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/CS221-NLP-Project-Vi-Fact-Checking-main/CS221-NLP-Project-Vi-Fact-Checking-main/notebooks/models/Phobert/phobert_model/runs/20260915_232534_858342/best_model
  History: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/CS221-NLP-Project-Vi-Fact-Checking-main/CS221-NLP-Project-Vi-Fact-Checking-main/notebooks/models/Phobert/phobert_model/runs/20260915_232534_858342/training_history.csv
  Predictions: /Users/nguyenhang/Document_Learning/TTNT/HK3/NLP/CS221-NLP-Project-Vi-Fact-Checking-main/CS221-NLP-Project

## 5. Tổng hợp file đầu ra và cách sử dụng

- `data/processed/phobert/`: dữ liệu đã xử lý URL, dữ liệu đã tách từ, tensor cho ba tập, mapping nhãn, tokenizer và `preprocessing_config.json`.
- `notebooks/models/Phobert/phobert_model/runs/<run_id>/best_model/`: trọng số mô hình, cấu hình mô hình, tokenizer và cấu hình tiền xử lý của thí nghiệm.
- `training_history.csv` trong mỗi thư mục run: loss và các metric theo epoch trên train/dev.
- `test_predictions.csv` trong mỗi thư mục run: nhãn thực tế và nhãn dự đoán trên test theo thứ tự dữ liệu đầu vào.

Khi suy luận, áp dụng cùng quy tắc xử lý URL, tách từ VnCoreNLP, tokenizer, độ dài tối đa và thứ tự Statement–Evidence đã dùng trong thí nghiệm. Mapping nhãn đi kèm checkpoint được dùng để diễn giải đầu ra số của mô hình.